<a href="https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

print("Warehouse connection ready")

Warehouse connection ready


## 1. Ranked actions + reason codes

The purpose of this queue is to help a human reviewer decide which content pages are worth looking at first.

I will use simple search signals rather than treating the score as an automatic decision.

### Reason codes

- `LOW_CTR_OPPORTUNITY` — the page has search impressions but very few clicks.
- `LOW_POSITION` — the page receives impressions but has a relatively weak average search position.
- `HIGH_OPPORTUNITY` — the page has meaningful search visibility and appears worth reviewing.

### Action labels

- `REVIEW_CTR` — review the page title and search snippet for a possible CTR improvement.
- `REVIEW_CONTENT` — review the content and search intent because the page has visibility but weak position.
- `MONITOR` — keep the page on the monitoring list instead of making an immediate change.

The score is only a prioritization score. It does not mean that a page definitely needs an update.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [3]:
# Avoid division by zero
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

df["action_score"] = (
    np.log1p(df["gsc_impressions"]) *
    (1 / (df["gsc_avg_position"].clip(lower=1)))
)

df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] >= 100) & (df["ctr"] < 0.02),
        (df["gsc_impressions"] >= 100) & (df["gsc_avg_position"] > 10),
    ],
    [
        "LOW_CTR_OPPORTUNITY",
        "LOW_POSITION"
    ],
    default="HIGH_OPPORTUNITY"
)

df["action"] = np.select(
    [
        df["reason_code"] == "LOW_CTR_OPPORTUNITY",
        df["reason_code"] == "LOW_POSITION"
    ],
    [
        "REVIEW_CTR",
        "REVIEW_CONTENT"
    ],
    default="MONITOR"
)

queue = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

queue = queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

queue.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,action_score,reason_code,action,rank
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,0.000025,10.598757,LOW_CTR_OPPORTUNITY,REVIEW_CTR,1
1,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,0.000000,10.415832,LOW_CTR_OPPORTUNITY,REVIEW_CTR,2
2,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,0.000000,10.403020,LOW_CTR_OPPORTUNITY,REVIEW_CTR,3
3,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,0.000061,10.396872,LOW_CTR_OPPORTUNITY,REVIEW_CTR,4
4,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.083407,0.000000,10.356885,LOW_CTR_OPPORTUNITY,REVIEW_CTR,5
5,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.117814,0.000032,10.340613,LOW_CTR_OPPORTUNITY,REVIEW_CTR,6
6,2026-03-24,client_23a62021009f63c4,content_44f34c0a90047651,30791,2,0.088955,0.000065,10.335010,LOW_CTR_OPPORTUNITY,REVIEW_CTR,7
7,2026-03-26,client_23a62021009f63c4,content_44f34c0a90047651,30573,2,0.238315,0.000065,10.327905,LOW_CTR_OPPORTUNITY,REVIEW_CTR,8
8,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,0.000000,10.274154,LOW_CTR_OPPORTUNITY,REVIEW_CTR,9
9,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,0.000000,10.273256,LOW_CTR_OPPORTUNITY,REVIEW_CTR,10


In [4]:
queue["reason_code"].value_counts()

,count
reason_code,
HIGH_OPPORTUNITY,2980206
LOW_CTR_OPPORTUNITY,628945
LOW_POSITION,1910


## 2. Intended use and limits

### Intended use

This playbook is intended to help a content reviewer prioritize pages for investigation.

The ranked queue can help answer:

- Which pages should I review first?
- Is the main signal a possible CTR opportunity or a position problem?
- Which pages should simply be monitored?

The output is decision-support, not an automatic publishing or editing system.

### Limits

The score does not prove that changing a page will improve traffic or rankings.

The current experiment uses March 2026 search-performance data and a same-period target. It therefore does not measure future causal impact.

The queue also does not contain information about content quality, search intent, business value, or whether a page is strategically important.

A human reviewer needs to consider those factors before taking action.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

Every recommended action should be reviewed by a person before any content change is made.

### Human review checklist

Before changing a page, the reviewer should check:

1. Does the page match the search intent?
2. Is the page still factually correct?
3. Is the title and snippet appropriate for the query?
4. Is the page important to the business?
5. Could the low clicks or position be caused by seasonality or another temporary factor?
6. Would changing the page create any legal, factual, or brand risk?

### What should NOT be automated

The system should not automatically:

- publish content changes;
- delete or redirect pages;
- change important business claims;
- change legal or regulated content;
- decide that a page is "bad";
- guarantee that an SEO change will improve performance;
- make decisions based only on the model score.

The score should only help decide what deserves human attention first.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers


The queue should be monitored rather than treated as a permanent rule.

### Monitoring

I would monitor:

- the number of pages receiving each reason code;
- the distribution of action scores;
- the proportion of reviewed pages that humans consider useful;
- changes in the input signals over time.

### Retrain or review triggers

I would review the model or rule if:

- the input data structure changes;
- the distribution of impressions or positions changes substantially;
- the model performance drops under the same validation method;
- the reason-code distribution changes unexpectedly;
- human reviewers frequently reject the highest-ranked recommendations.

These triggers are signals to investigate, not automatic proof that the model has failed.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

The notebook exports the ranked action queue so that the same output can be used when writing the recommendations section of the research paper.

The CSV is regenerated by the notebook rather than treated as a manually maintained dataset.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(
    output_path,
    index=False
)

print("Exported:", output_path)
print("Rows:", len(queue))

Exported: /content/work/outputs/baseline_action_score.csv
Rows: 3611061


In [9]:
check = pd.read_csv(output_path)

print("File exists:", os.path.exists(output_path))
print("Rows exported:", len(check))
print("Columns:")
print(check.columns.tolist())

check.head(10)

File exists: True
Rows exported: 3611061
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'action_score', 'reason_code', 'action', 'rank']


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,action_score,reason_code,action,rank
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,0.000025,10.598757,LOW_CTR_OPPORTUNITY,REVIEW_CTR,1
1,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,0.000000,10.415832,LOW_CTR_OPPORTUNITY,REVIEW_CTR,2
2,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,0.000000,10.403020,LOW_CTR_OPPORTUNITY,REVIEW_CTR,3
3,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,0.000061,10.396872,LOW_CTR_OPPORTUNITY,REVIEW_CTR,4
4,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.083407,0.000000,10.356885,LOW_CTR_OPPORTUNITY,REVIEW_CTR,5
5,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.117814,0.000032,10.340613,LOW_CTR_OPPORTUNITY,REVIEW_CTR,6
6,2026-03-24,client_23a62021009f63c4,content_44f34c0a90047651,30791,2,0.088955,0.000065,10.335010,LOW_CTR_OPPORTUNITY,REVIEW_CTR,7
7,2026-03-26,client_23a62021009f63c4,content_44f34c0a90047651,30573,2,0.238315,0.000065,10.327905,LOW_CTR_OPPORTUNITY,REVIEW_CTR,8
8,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,0.000000,10.274154,LOW_CTR_OPPORTUNITY,REVIEW_CTR,9
9,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,0.000000,10.273256,LOW_CTR_OPPORTUNITY,REVIEW_CTR,10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.